# DATALORA Training

1. ZIP dosyasını yükle
2. A100 GPU seç
3. Hücreleri sırayla çalıştır

In [ ]:
#@title ADIM 1: ZIP aç ve kur
%cd /content
!rm -rf datalora
!unzip -o datalora_project.zip
!pip install -q torch transformers accelerate peft datasets bitsandbytes wandb einops scipy

In [ ]:
#@title ADIM 2: Import
from datasets import load_dataset

import sys
sys.path.insert(0, '/content/datalora/src')

import torch
from transformers import AutoTokenizer, DataCollatorForLanguageModeling

from models import DATALORAConfig, load_datalora_model
from training import DATALORATrainer, DATALORATrainingArguments

print("Import OK")

In [ ]:
#@title ADIM 3: HuggingFace Login
from huggingface_hub import login
login()

In [ ]:
#@title ADIM 4: Config ve Tokenizer
BASE_MODEL = "mistralai/Mistral-7B-v0.3"

config = DATALORAConfig(
    base_model=BASE_MODEL,
    num_lora_experts=8,
    lora_rank=16,
    target_retention=0.5,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token

print("Config ve Tokenizer OK")

In [ ]:
#@title ADIM 5: Model Yükle
print("Model yükleniyor...")

model = load_datalora_model(
    base_model_name=BASE_MODEL,
    config=config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    load_in_4bit=True,
)

print("Model OK")

In [ ]:
#@title ADIM 6: Dataset
dataset = load_dataset("Open-Orca/OpenOrca", split="train")
dataset = dataset.select(range(5000))

def tokenize_fn(examples):
    texts = [f"<|user|>\n{q}\n<|assistant|>\n{r}" for q, r in zip(examples['question'], examples['response'])]
    tok = tokenizer(texts, truncation=True, max_length=512, padding="max_length")
    tok["labels"] = tok["input_ids"].copy()
    return tok

tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=dataset.column_names)
train_data = tokenized.select(range(4500))
eval_data = tokenized.select(range(4500, 5000))

print(f"Train: {len(train_data)}, Eval: {len(eval_data)}")

In [ ]:
#@title ADIM 7: Trainer
training_args = DATALORATrainingArguments(
    output_dir="/content/outputs/datalora",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    logging_steps=25,
    save_strategy="epoch",
    bf16=True,
    report_to=["wandb"],
    warmup_epochs=1,
    sparsification_epochs=1,
    hardening_epochs=1,
)

trainer = DATALORATrainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=eval_data,
    tokenizer=tokenizer,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

print("Trainer OK")

In [ ]:
#@title ADIM 8: Eğitim
trainer.train()
trainer.save_model()
print("Eğitim tamamlandı!")